# Monotonicity Metric Variants: Calculation & Failure Cases

This notebook answers two concrete questions:
1. **How is each metric variant actually computed?** — step-by-step.
2. **Where is two-sided minimum definitively worse than another measure?**

Run from `src/examples/learn_quant/` with the `altk` conda environment.

## 1 · The four monotone approximations

Every metric is built from four canonical monotone functions that best approximate Q.

| Symbol | Name | What it asks for every situation x |
| ------ | ---- | ------------------------------------ |
| C_up(Q)(x) | least upward majorant | Does x have a true predecessor (wrt B-expansion)? |
| I_up(Q)(x) | greatest upward minorant | Are ALL successors of x true? |
| C_down(Q)(x) | least downward majorant | Does x have a true successor? |
| I_down(Q)(x) | greatest downward minorant | Are ALL predecessors of x true? |

**Key identity** — the code computes all four using only `has_true_predecessor` + NOT Q:

```
I_up(Q)   = NOT C_down(NOT Q)   # interior = complement of majorant of complement
I_down(Q) = NOT C_up(NOT Q)
```

### The entropy score

Once we have an approximation F of Q, the score is:
```
score(Q, F) = 1 - H(Q | pred_F) / H(Q)
```
where `pred_F(x) = 1` iff x has a F-true predecessor.

- If Q is exactly monotone: pred and Q agree everywhere, H(Q|pred)=0, score=1.
- If Q is random: pred tells you nothing new, H(Q|pred)=H(Q), score=0.

### How each variant combines them

| Variant | Right-upward | Right-downward |
| ------- | ------------ | -------------- |
| **Majorant** | score( C_up_B(Q) ) | score( C_down_B(Q) ) |
| **Complement-dual** | score( C_up_B(Q) ) | score( C_up_B(not Q) ) |
| **Two-sided mean** | mean( C_up_B, I_up_B ) | mean( C_down_B, I_down_B ) |
| **Two-sided min** | **min( C_up_B, I_up_B )** | min( C_down_B, I_down_B ) |
| **Violation rate** | 1 - T-to-F_transitions / all_comparable_pairs | 1 - F-to-T / ... |

> The key difference between majorant and two-sided min is that min also requires the
> **interior score** (I_up) to be high — i.e. ALL successors of a true model must also be true.
> This is a much stricter criterion and it collapses for many non-exact predicates.

## 2 · Worked example: `A != B` on a tiny M2/X2 universe

We build the universe manually so you can inspect every situation.

In [1]:
import sys, numpy as np, pandas as pd, pathlib
from itertools import product

# M2/X2: A in subsets of {a1,a2}, B in subsets of {b1,b2}  ->  16 situations
situations = list(product(range(4), range(4)))  # (a_bits, b_bits)

def preds_right(a, b, sits):
    """Predecessors under B-expansion: same A, B_other is a subset of B."""
    return [(aa, bb) for aa, bb in sits if aa == a and (bb & b) == bb]

def succs_right(a, b, sits):
    """Successors under B-expansion: same A, B is a subset of B_other."""
    return [(aa, bb) for aa, bb in sits if aa == a and (b & bb) == b]

Q_neq = np.array([int(a != b) for a, b in situations])

C_up_vec = np.array([
    int(any(Q_neq[situations.index(p)] for p in preds_right(a, b, situations)))
    for a, b in situations
])
I_up_vec = np.array([
    int(all(Q_neq[situations.index(s)] for s in succs_right(a, b, situations)))
    for a, b in situations
])

print('  A   B  | Q  | C_up | I_up')
print('-' * 35)
for i, (a, b) in enumerate(situations):
    print(f'  {a:02b} {b:02b}  |  {Q_neq[i]}  |   {C_up_vec[i]}    |   {I_up_vec[i]}')
print()
print(f'Q prevalence: {Q_neq.mean():.0%}  (true when A-bits != B-bits)')
print(f'I_up=1 for only {I_up_vec.mean():.0%} of all situations')

  A   B  | Q  | C_up | I_up
-----------------------------------
  00 00  |  0  |   0    |   0
  00 01  |  1  |   1    |   1
  00 10  |  1  |   1    |   1
  00 11  |  1  |   1    |   1
  01 00  |  1  |   1    |   0
  01 01  |  0  |   1    |   0
  01 10  |  1  |   1    |   1
  01 11  |  1  |   1    |   1
  10 00  |  1  |   1    |   0
  10 01  |  1  |   1    |   1
  10 10  |  0  |   1    |   0
  10 11  |  1  |   1    |   1
  11 00  |  1  |   1    |   0
  11 01  |  1  |   1    |   0
  11 10  |  1  |   1    |   0
  11 11  |  0  |   1    |   0

Q prevalence: 75%  (true when A-bits != B-bits)
I_up=1 for only 44% of all situations


In [2]:
# Compute entropy scores for C_up vs I_up
def entropy_score(Q, pred):
    Q, pred = np.asarray(Q, float), np.asarray(pred, float)
    p_q = Q.mean()
    if p_q in (0.0, 1.0):
        return 1.0
    h_q = -p_q * np.log2(p_q) - (1 - p_q) * np.log2(1 - p_q)

    def safe_xlogx(p, denom):
        if denom == 0 or p == 0:
            return 0.0
        return -p * np.log2(p / denom)

    p_pp = pred.mean()
    p_pn = 1 - p_pp
    pp_T = (Q * pred).mean()
    pp_F = ((1 - Q) * pred).mean()
    pn_T = (Q * (1 - pred)).mean()
    pn_F = ((1 - Q) * (1 - pred)).mean()
    h_cond = (
        safe_xlogx(pp_T, p_pp) + safe_xlogx(pp_F, p_pp)
        + safe_xlogx(pn_T, p_pn) + safe_xlogx(pn_F, p_pn)
    )
    return 1 - h_cond / h_q

sc_cup = entropy_score(Q_neq, C_up_vec)
sc_iup = entropy_score(Q_neq, I_up_vec)

print('A != B  (M2/X2, right direction / B-expansion):')
print(f'  score(C_up) = {sc_cup:.3f}   <- majorant score')
print(f'  score(I_up) = {sc_iup:.3f}   <- interior score')
print(f'  two-sided min  = {min(sc_cup, sc_iup):.3f}')
print(f'  two-sided mean = {(sc_cup + sc_iup) / 2:.3f}')
print()
print('Why does I_up collapse?')
print('  For a true model (a != b), expand B until B=A: that successor is FALSE.')
print('  => I_up=False for almost every true model.')
print('  => the interior score collapses, and min(C_up, I_up) collapses with it.')

A != B  (M2/X2, right direction / B-expansion):
  score(C_up) = 0.166   <- majorant score
  score(I_up) = 0.313   <- interior score
  two-sided min  = 0.166
  two-sided mean = 0.239

Why does I_up collapse?
  For a true model (a != b), expand B until B=A: that successor is FALSE.
  => I_up=False for almost every true model.
  => the interior score collapses, and min(C_up, I_up) collapses with it.


## 3 · Loading the M6/X6 benchmark results

In [3]:
# notebooks/ lives inside learn_quant/ so go up one level to find analysis/
REPO = pathlib.Path(__file__).parent.parent if '__file__' in dir() else pathlib.Path('..')
df = pd.read_csv(REPO / 'analysis/semantic_monotonicity_benchmark.csv')

dirs = ['RU', 'LU', 'RD', 'LD']
rows = []
for _, r in df.iterrows():
    for d in dirs:
        rows.append({
            'meaning': r['meaning'],
            'formula': r['formula'],
            'exact': d in str(r['expected_exact_directions']),
            'dir': d,
            'min': r[f'min_{d}'],
            'majorant': r[f'majorant_{d}'],
            'mean': r[f'mean_{d}'],
            'pres': r[f'preservation_{d}'],
        })
L = pd.DataFrame(rows)

# Cases where two-sided min diverges most from the best alternative
ne = L[~L['exact']].copy()
ne['gap_from_best'] = ne[['majorant', 'mean', 'pres']].max(axis=1) - ne['min']
top = (ne.sort_values('gap_from_best', ascending=False)
         .drop_duplicates('meaning')
         .head(10)
         [['meaning', 'dir', 'min', 'majorant', 'mean', 'pres', 'gap_from_best']]
         .round(3))
print('Top 10 non-exact meanings where two-sided min lags farthest behind:')
display(top)

Top 10 non-exact meanings where two-sided min lags farthest behind:


,meaning,dir,min,majorant,mean,pres,gap_from_best
39,A differs from B,LD,0.013,0.013,0.177,0.984,0.971
111,both A and B are nonempty,LD,0.000,0.427,0.213,0.905,0.905
123,A and B are both empty or both nonempty,LD,0.000,0.423,0.212,0.905,0.905
69,different numbers of A and B,LU,0.001,0.001,0.123,0.766,0.766
105,at most half of A are B,LU,0.000,0.000,0.144,0.765,0.765
97,at least half of A are B,LU,0.000,0.000,0.144,0.765,0.765
4,not all A are B,RU,0.000,0.059,0.029,0.706,0.706
23,some A are not B,LD,0.000,0.059,0.029,0.706,0.706
15,some A are B,LD,0.000,0.059,0.029,0.706,0.706
44,A and B are incomparable,RU,0.000,0.403,0.201,0.673,0.673


## 4 · Four named failure cases

### Case A: `A != B` — two-sided min undersells a near-tautology

| | min | majorant | mean | preservation |
| - | --- | -------- | ---- | ------------ |
| RU score | **0.013** | 0.013 | 0.177 | **0.984** |

A != B is true in 98.4% of M6/X6 situations.  
**Preservation** says: out of all comparable pairs where Q(smaller)=True,
only 1.6% flip to False. This is the most intuitive reading.

**Why min (and majorant) give 0.013:** The entropy calculation is dominated by the
marginal distribution. The 1.6% false (A=B) models contaminate the pred=1 bucket
because they DO have true predecessors (expand one set and you leave equality).
Those false models inside the pred=1 bucket inflate H(Q|pred=1), lowering the score.
The interior score collapses further because true models always have an expansion path
back to A=B.

**Verdict:** Use preservation/violation-rate when you want 'how often does a
direction-expansion cause a violation?'  Entropy metrics are biased by prevalence.

---

### Case B: `A and B are incomparable` — two-sided min loses the approximation signal

| | min | majorant | preservation |
| - | --- | -------- | ------------ |
| RU score | **0.000** | **0.403** | 0.673 |

Incomparable is not exactly monotone, yet the majorant gives 0.403.
That score has a real meaning: knowing whether an incomparable predecessor exists
reduces entropy about Q by 40%. This is genuine approximation information.

**Why min = 0:** The interior I_up = 0 for every model. From any incomparable (A,B),
you can expand B far enough to make A <= B (comparable, False). The interior requires
ALL successors to be true — this always fails. min(0.403, 0) = 0.

**Verdict:** Use the majorant when you want 'how well can Q be approximated from above
by a monotone function?'  Two-sided min destroys that signal.

---

### Case C: `exactly five overlap` — two-sided min gives a false positive

| | min | majorant | preservation |
| - | --- | -------- | ------------ |
| RU score | **0.610** | 0.966 | **0.500** |

This predicate is NOT monotone, yet min = 0.610.
On M6/X6, the only upward violation of |A intersect B|=5 is the 5->6 transition.
That transition is rare (requires exactly M=X=6 elements). Both C_up and I_up score
high because the violation is a single boundary event. Min bottlenecks at 0.610.

**Preservation = 0.500** is most honest: exactly half of the rare boundary transitions
violate upward monotonicity. On an unbounded domain this score would collapse to near zero.

**Verdict:** Finite-universe boundary effects inflate entropy metrics. Preservation is
more calibrated. Always report the universe size alongside boundary predicates.

---

### Case D: `A and B both empty or both nonempty` — every metric fails

| | min | majorant | preservation |
| - | --- | -------- | ------------ |
| RU score | **0.423** | 0.423 | **0.998** |

96.9% of situations satisfy this (most sets are nonempty). It is NOT monotone:
the T(both empty) -> F(one empty) transition exists.

**Why preservation = 0.998 is wrong:** The violations only appear among the 3.1% of
situations where one set is empty. Comparable pairs involving empty sets are rare,
so violation rate is near zero even though the function is not monotone.

**Why min = 0.423 is less wrong:** The entropy calculation detects the boundary
structure around the empty set, giving a non-trivial score.

**Verdict:** When prevalence is extreme (near 0 or 1), all metrics can mislead.
Neither metric reliably detects monotonicity violations here.

In [4]:
cases = [
    'A differs from B',
    'A and B are incomparable',
    'exactly five overlap',
    'A and B are both empty or both nonempty',
]
for m in cases:
    r = L[(L['meaning'] == m) & (L['dir'] == 'RU')].iloc[0]
    print(f'{m}:')
    print(f'  min={r["min"]:.3f}  maj={r["majorant"]:.3f}  mean={r["mean"]:.3f}  pres={r["pres"]:.3f}')
    print()

A differs from B:
  min=0.013  maj=0.013  mean=0.177  pres=0.984

A and B are incomparable:
  min=0.000  maj=0.403  mean=0.201  pres=0.673

exactly five overlap:
  min=0.610  maj=0.966  mean=0.788  pres=0.500

A and B are both empty or both nonempty:
  min=0.423  maj=0.423  mean=0.706  pres=0.998



## 5 · Summary: when to use each metric

| Question | Best metric | Why it wins |
| -------- | ----------- | ----------- |
| Does this predicate have **exact** monotonicity? | Two-sided min | =1 iff exact; no false positives on exact predicates |
| How well can Q be **approximated** by a monotone function? | Majorant | Captures closure approximation quality |
| How often does an expansion create a **direction violation**? | Preservation / violation rate | Directly counts T-to-F transitions |
| Do **complement pairs** get mirror-symmetric scores? | Two-sided min | 44/44 complement pairs: zero error |
| High-prevalence or **near-tautology** predicates? | Violation rate | Entropy metrics biased by marginal distribution |
| **Boundary / exactly-N** predicates on finite universe? | None reliable | Score depends on domain; report universe size |

No single metric wins everywhere. The manuscript majorant is appropriate for measuring
approximability of grammar-generated expressions. Two-sided min is best for
complement-symmetric analyses. Preservation is most interpretable as a semantic check.